<span style="font-size:11px">

#### >>> 회귀
- 출력의 개수를 1개로
- 손실함수는 MSE나 기타 등등..
- 데이터셋과 데이터로드를 커스텀하게 정의해서 사용
- 나머지는 동일한 패턴으로 학습/평가

In [ ]:
import pandas as pd
import numpy as np

data_url = "http://lib.stat.cmu.edu/datasets/boston"
raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)
data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
target = raw_df.values[1::2, 2]

((506, 13), (506,))

In [3]:
data.shape, target.shape

((506, 13), (506,))

In [21]:
from torch.utils.data import Dataset,DataLoader
import pandas as pd
import numpy as np
import torch


data_url = "http://lib.stat.cmu.edu/datasets/boston"
raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)
data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
target = raw_df.values[1::2, 2]


class BostonDataSet (Dataset):
    def __init__(self, X,y):


        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1,1)
    def __len__(self):
        return len(self.X)
    def __getitem__(self,idx):
        return self.X[idx], self.y[idx]


In [22]:
X_dataset = BostonDataSet(data, target) #dataset은 데이터 X,y를 관리할수 있게 한쌍으로 묶어줌.
X_train_loader =DataLoader(X_dataset, batch_size=32, shuffle=True)

In [23]:
# 회귀 모델 정의
import torch.nn as nn

class BostonRegression(nn.Module):
    def __init__ (self, input_dim ):
        super(BostonRegression,self).__init__()
        self.model= nn.Sequential(
            nn.Linear(input_dim,64),
            nn.ReLU(),
            nn.Linear(64,32),
            nn.ReLU(),
            nn.Linear(32,1)
        ) 
    def forward(self, X):
        return self.model(X)
    

In [19]:
from torch.optim import Adam
model = BostonRegression(data.shape[1])
criterion = nn.MSELoss()
optim= Adam(model.parameters(), lr=1e-3)


In [24]:
######실행3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
epochs = 500


from tqdm import tqdm 
# 학습루프
for epoch in range(epochs):
    tqdm_obj= tqdm(X_train_loader, desc=f'epoch: {epoch+1}/{epochs}')
    loss_lists = 0
    for data, label in tqdm_obj:
        optim.zero_grad()
        preds = model (data.to(device))
        loss = criterion(preds, label.to(device))
        loss_lists += loss.item()
        loss.backward()
        optim.step()
        tqdm_obj.set_postfix({'loss' : f'{loss.item():.4f}'})

    avg_loss = loss_lists / len(X_train_loader)
    print(f'epoch : {epoch+1}, avg loss: {avg_loss:.4f}')
    

torch.save(model.state_dict(), 'bostonRegression.pth')  #torch에는 모델 저장 기능이 있음 / 가중치만 저장되어있음./ 모델구조는 저장되어 있지 않음. (모델+가중치 저장되는 것도 있음. 용량 많이 차지함)
        


epoch: 1/500: 100%|██████████| 16/16 [00:00<00:00, 300.72it/s, loss=3.3567]


epoch : 1, avg loss: 9.1610


epoch: 2/500: 100%|██████████| 16/16 [00:00<00:00, 331.43it/s, loss=7.4561]


epoch : 2, avg loss: 9.9084


epoch: 3/500: 100%|██████████| 16/16 [00:00<00:00, 379.86it/s, loss=8.2256]


...

epoch : 498, avg loss: 5.7157


epoch: 499/500: 100%|██████████| 16/16 [00:00<00:00, 255.42it/s, loss=3.5890]


epoch : 499, avg loss: 5.3892


epoch: 500/500: 100%|██████████| 16/16 [00:00<00:00, 341.67it/s, loss=7.7294]

epoch : 500, avg loss: 4.9012


In [25]:
# 평가
model.load_state_dict(torch.load('bostonRegression.pth',map_location=device,weights_only=True))

model.eval()  # 평가 모드로 전환 (dropout, batchnorm 등 비활성화)
total_mse = 0

criterion = nn.MSELoss()
with torch.no_grad():  # 그래디언트 계산 비활성화
    for data, label in tqdm(X_train_loader, desc="Evaluating"):
        data, label = data.to(device), label.to(device)
        preds = model(data)
        mse = criterion(preds, label)
        total_mse += mse.item() * data.size(0)  

print(f"Test Loss: {total_mse/len(X_train_loader)}")  #accuracy는????

Evaluating: 100%|██████████| 16/16 [00:00<00:00, 2192.74it/s]

Test Loss: 129.21128398180008
